# PyTorch to TensorRT Quantization & Visualization Workflow

This notebook demonstrates:
1. Training a CNN on MNIST
2. Converting to TensorRT with FP32, FP16, and INT8 precision
3. Benchmarking performance and accuracy
4. Exporting for Netron visualization

## Prerequisites
```bash
pip install torch torchvision onnx torch-tensorrt --break-system-packages
```

## Cell 1: Dataset & Model Definition + Training

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import time
import os

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Define a simple CNN (Mini-ResNet style)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # First block
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        
        # Second block
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)
        
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Block 1
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        
        # Block 2
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        
        # Classifier
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('../data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Using device: cuda
Training samples: 60000
Test samples: 10000


In [18]:
# Training function
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    correct = 0
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = nn.CrossEntropyLoss()(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} '
                  f'({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}')
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / len(train_loader.dataset)
    print(f'\nTraining Epoch {epoch}: Avg Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%\n')
    return avg_loss, accuracy

# Evaluation function
def evaluate(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += nn.CrossEntropyLoss(reduction='sum')(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    
    print(f'Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} '
          f'({accuracy:.2f}%)\n')
    return test_loss, accuracy

In [19]:
# Train the model
model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(1, num_epochs + 1):
    train(model, device, train_loader, optimizer, epoch)
    evaluate(model, device, test_loader)

# Save the trained model
os.makedirs('../models', exist_ok=True)
torch.save(model.state_dict(), '../models/mnist_cnn.pth')
print("Model saved to ../models/mnist_cnn.pth")

Epoch: 1 [0/60000 (0%)]	Loss: 2.393115
Epoch: 1 [12800/60000 (21%)]	Loss: 0.199449
Epoch: 1 [25600/60000 (43%)]	Loss: 0.095292
Epoch: 1 [38400/60000 (64%)]	Loss: 0.073010
Epoch: 1 [51200/60000 (85%)]	Loss: 0.061432

Training Epoch 1: Avg Loss: 0.1914, Accuracy: 94.23%

Test set: Average loss: 0.0387, Accuracy: 9875/10000 (98.75%)

Epoch: 2 [0/60000 (0%)]	Loss: 0.071371
Epoch: 2 [12800/60000 (21%)]	Loss: 0.113670
Epoch: 2 [25600/60000 (43%)]	Loss: 0.200141
Epoch: 2 [38400/60000 (64%)]	Loss: 0.015917
Epoch: 2 [51200/60000 (85%)]	Loss: 0.045398

Training Epoch 2: Avg Loss: 0.0750, Accuracy: 97.84%

Test set: Average loss: 0.0304, Accuracy: 9892/10000 (98.92%)

Epoch: 3 [0/60000 (0%)]	Loss: 0.041510
Epoch: 3 [12800/60000 (21%)]	Loss: 0.029373
Epoch: 3 [25600/60000 (43%)]	Loss: 0.093243
Epoch: 3 [38400/60000 (64%)]	Loss: 0.036857
Epoch: 3 [51200/60000 (85%)]	Loss: 0.061837

Training Epoch 3: Avg Loss: 0.0565, Accuracy: 98.34%

Test set: Average loss: 0.0226, Accuracy: 9928/10000 (99.28%)

E

## Cell 2: TensorRT Optimization & Quantization

We'll convert the model to:
1. **FP32** (baseline)
2. **FP16** (half precision)
3. **INT8** (with calibration)

In [20]:
import torch_tensorrt
import numpy as np

# Load the trained model
model = SimpleCNN().to(device)
model.load_state_dict(torch.load('../models/mnist_cnn.pth'))
model.eval()

# Prepare example input for TensorRT compilation
example_input = torch.randn(1, 1, 28, 28).to(device)

print("Starting TensorRT compilation...\n")

Starting TensorRT compilation...



In [21]:
# 1. FP32 TensorRT Model (Baseline)
print("Compiling FP32 model...")

# First, trace the model to TorchScript
traced_model = torch.jit.trace(model, torch.randn(1, 1, 28, 28).cuda())

# Then compile with TensorRT
trt_model_fp32 = torch_tensorrt.compile(
    traced_model,  # Use traced model instead
    inputs=[torch_tensorrt.Input(shape=[1, 1, 28, 28])],
    enabled_precisions={torch.float32},
    workspace_size=1 << 30  # 1GB
)

# Now save works
torch.jit.save(trt_model_fp32, '../models/mnist_trt_fp32.ts')
print("FP32 model saved\n")

Compiling FP32 model...


FP32 model saved



In [22]:
# 2. FP16 TensorRT Model
print("Compiling FP16 model...")
traced_model_fp16 = torch.jit.trace(model, torch.randn(1, 1, 28, 28).cuda())
trt_model_fp16 = torch_tensorrt.compile(
    traced_model_fp16,
    inputs=[torch_tensorrt.Input(shape=[1, 1, 28, 28])],
    enabled_precisions={torch.half},  # FP16
    workspace_size=1 << 30
)
torch.jit.save(trt_model_fp16, '../models/mnist_trt_fp16.ts')
print("FP16 model saved\n")

Compiling FP16 model...


FP16 model saved



In [23]:
# 3. INT8 TensorRT Model with Calibration
print("Preparing INT8 calibration...")

# Create calibration dataset (subset of training data)
calibration_size = 1000
calibration_indices = np.random.choice(len(train_dataset), calibration_size, replace=False)
calibration_dataset = Subset(train_dataset, calibration_indices)
calibration_loader = DataLoader(calibration_dataset, batch_size=1, shuffle=False)

print("Compiling INT8 model with calibration...")

# Trace the model first
traced_model_int8 = torch.jit.trace(model, torch.randn(1, 1, 28, 28).cuda())

# Compile with INT8 (calibration happens automatically)
trt_model_int8 = torch_tensorrt.compile(
    traced_model_int8,
    inputs=[torch_tensorrt.Input(shape=[1, 1, 28, 28])],
    enabled_precisions={torch.int8},
    workspace_size=1 << 30
)

torch.jit.save(trt_model_int8, '../models/mnist_trt_int8.ts')
print("INT8 model saved\n")

print("All TensorRT models compiled successfully!")

Preparing INT8 calibration...
Compiling INT8 model with calibration...
INT8 model saved

All TensorRT models compiled successfully!


## Cell 3: Inference & Benchmarking

Compare performance across all model versions

In [24]:
import pandas as pd
from pathlib import Path

# Benchmark function
def benchmark_model(model, test_loader, device, model_name, num_warmup=10, num_runs=100):
    model.eval()
    correct = 0
    total = 0
    
    # Warmup
    print(f"Warming up {model_name}...")
    with torch.no_grad():
        for i, (data, _) in enumerate(test_loader):
            if i >= num_warmup:
                break
            data = data.to(device)
            _ = model(data)
    
    # Benchmark inference time
    print(f"Benchmarking {model_name}...")
    inference_times = []
    
    with torch.no_grad():
        for i, (data, target) in enumerate(test_loader):
            if i >= num_runs:
                break
            
            data, target = data.to(device), target.to(device)
            
            # Time inference
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_time = time.time()
            
            output = model(data)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_time = time.time()
            
            inference_times.append(end_time - start_time)
            
            # Calculate accuracy
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
    
    avg_inference_time = np.mean(inference_times) * 1000  # Convert to ms
    std_inference_time = np.std(inference_times) * 1000
    throughput = 1000 / avg_inference_time  # Images per second (batch size 1000)
    accuracy = 100. * correct / total
    
    return {
        'avg_time_ms': avg_inference_time,
        'std_time_ms': std_inference_time,
        'throughput': throughput,
        'accuracy': accuracy
    }

# Get model sizes
def get_model_size(filepath):
    if Path(filepath).exists():
        size_bytes = Path(filepath).stat().st_size
        size_mb = size_bytes / (1024 * 1024)
        return size_mb
    return None

In [25]:
# Load all models
print("Loading models...\n")

# PyTorch native model
model_pytorch = SimpleCNN().to(device)
model_pytorch.load_state_dict(torch.load('../models/mnist_cnn.pth'))
model_pytorch.eval()

# TensorRT models
model_trt_fp32 = torch.jit.load('../models/mnist_trt_fp32.ts').to(device)
model_trt_fp16 = torch.jit.load('../models/mnist_trt_fp16.ts').to(device)
model_trt_int8 = torch.jit.load('../models/mnist_trt_int8.ts').to(device)

print("All models loaded!\n")

Loading models...

All models loaded!



In [26]:
# Run benchmarks
results = {}

print("="*60)
print("BENCHMARKING ALL MODELS")
print("="*60)

# Create a special test loader with batch_size=1 for TensorRT models
test_loader_bs1 = DataLoader(test_dataset, batch_size=1, shuffle=False)

models_to_test = [
    ('PyTorch Native', model_pytorch, '../models/mnist_cnn.pth', test_loader),
    ('TensorRT FP32', model_trt_fp32, '../models/mnist_trt_fp32.ts', test_loader_bs1),
    ('TensorRT FP16', model_trt_fp16, '../models/mnist_trt_fp16.ts', test_loader_bs1),
    ('TensorRT INT8', model_trt_int8, '../models/mnist_trt_int8.ts', test_loader_bs1),
]

for name, model, filepath, loader in models_to_test:
    print(f"\n{name}:")
    result = benchmark_model(model, loader, device, name)
    result['model_size_mb'] = get_model_size(filepath)
    results[name] = result
    print(f"  Avg Time: {result['avg_time_ms']:.2f} ± {result['std_time_ms']:.2f} ms")
    print(f"  Throughput: {result['throughput']:.2f} batches/sec")
    print(f"  Accuracy: {result['accuracy']:.2f}%")
    if result['model_size_mb']:
        print(f"  Model Size: {result['model_size_mb']:.2f} MB")

BENCHMARKING ALL MODELS

PyTorch Native:
Warming up PyTorch Native...
Benchmarking PyTorch Native...
  Avg Time: 63.49 ± 15.26 ms
  Throughput: 15.75 batches/sec
  Accuracy: 99.05%
  Model Size: 1.80 MB

TensorRT FP32:
Warming up TensorRT FP32...
Benchmarking TensorRT FP32...
  Avg Time: 1.37 ± 1.99 ms
  Throughput: 731.78 batches/sec
  Accuracy: 100.00%
  Model Size: 2.95 MB

TensorRT FP16:
Warming up TensorRT FP16...
Benchmarking TensorRT FP16...
  Avg Time: 0.43 ± 1.15 ms
  Throughput: 2332.01 batches/sec
  Accuracy: 100.00%
  Model Size: 2.59 MB

TensorRT INT8:
Warming up TensorRT INT8...
Benchmarking TensorRT INT8...
  Avg Time: 0.57 ± 1.26 ms
  Throughput: 1743.09 batches/sec
  Accuracy: 100.00%
  Model Size: 2.93 MB


In [27]:
# Create comparison table
print("\n" + "="*80)
print("PERFORMANCE COMPARISON TABLE")
print("="*80)

df = pd.DataFrame(results).T
df = df.round(2)

# Calculate speedup relative to PyTorch native
baseline_time = results['PyTorch Native']['avg_time_ms']
df['speedup'] = baseline_time / df['avg_time_ms']
df['speedup'] = df['speedup'].round(2)

# Reorder columns
df = df[['accuracy', 'avg_time_ms', 'std_time_ms', 'speedup', 'throughput', 'model_size_mb']]
df.columns = ['Accuracy (%)', 'Avg Time (ms)', 'Std Time (ms)', 'Speedup', 'Throughput (batch/s)', 'Size (MB)']

print(df.to_string())
print("\n" + "="*80)

# Save to CSV
df.to_csv('../results/benchmark_results.csv')
print("\nResults saved to ../results/benchmark_results.csv")


PERFORMANCE COMPARISON TABLE
                Accuracy (%)  Avg Time (ms)  Std Time (ms)  Speedup  Throughput (batch/s)  Size (MB)
PyTorch Native         99.05          63.49          15.26     1.00                 15.75       1.80
TensorRT FP32         100.00           1.37           1.99    46.34                731.78       2.95
TensorRT FP16         100.00           0.43           1.15   147.66               2332.01       2.59
TensorRT INT8         100.00           0.57           1.26   111.39               1743.09       2.93


Results saved to ../results/benchmark_results.csv


## Cell 4: Export for Netron Visualization

Export the model to ONNX format for visualization in [Netron](https://netron.app)

In [28]:
import onnx

# Load the PyTorch model
model = SimpleCNN().to('cpu')  # Move to CPU for ONNX export
model.load_state_dict(torch.load('../models/mnist_cnn.pth', map_location='cpu'))
model.eval()

# Create dummy input
dummy_input = torch.randn(1, 1, 28, 28)

# Export to ONNX
onnx_path = '../models/mnist_cnn.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model exported to ONNX: {onnx_path}")

# Verify the ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model verification passed!")

print(f"\nModel info:")
print(f"   - Input: {onnx_model.graph.input[0].name}")
print(f"   - Output: {onnx_model.graph.output[0].name}")
print(f"   - Nodes: {len(onnx_model.graph.node)}")

  torch.onnx.export(

W0129 09:53:00.703000 18916 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "c:\codes\testPytorch\env1\Lib\site-packages\onnxscript\version_converter\__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\codes\testPytorch\env1\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "c:\codes\testPytorch\env1\Lib\site-packages\onnxscript\version_converter\__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\codes\testPytorch\env1\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: D:\a\onnx\onnx\onnx/version_converter/BaseConverter.h:68: adapter_lookup: Asse

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 5 of general pattern rewrite rules.
Model exported to ONNX: ../models/mnist_cnn.onnx
ONNX model verification passed!

Model info:
   - Input: input
   - Output: output
   - Nodes: 16
